# LeetCode 784 — Letter Case Permutation bằng Python

Link đề: https://leetcode.com/problems/letter-case-permutation/ · Độ khó: **Medium** · Chủ đề: Bit Manipulation, String, Backtracking

Skeleton LeetCode:

```python
class Solution:
    def letterCasePermutation(self, s: str) -> List[str]:
        ...
```

Ghi chú: đề cho phép trả kết quả theo **thứ tự bất kỳ**, nên khi test ta **so sánh sau khi sắp xếp** (`sorted`) và dùng thêm hàm kiểm tra tính chất (đủ `2^k` chuỗi khác nhau, mỗi chuỗi đúng dạng).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với chuỗi `s` gồm chữ cái và chữ số, **mỗi chữ cái** có thể được đổi thành chữ **thường** hoặc **hoa** một cách độc lập; liệt kê **mọi chuỗi** có thể tạo ra (thứ tự tùy ý).
2. Nắm các công cụ Python: `str.isalpha`, `str.lower/upper/swapcase`, `ord/chr` và mẹo **XOR với `32`** để đổi hoa/thường, chuỗi **bất biến** nên dùng `list` rồi `"".join`, đệ quy quay lui, thao tác bit (`1 << k`, `&`, `>>`), và `itertools.product`.
3. Đi từ **thử mọi mặt nạ bit cho mọi ký tự rồi khử trùng** → **DFS quay lui** → **mở rộng dần danh sách kết quả (BFS)** → **mặt nạ bit chỉ trên các chữ cái** → **mã Gray (mỗi bước lật đúng một chữ cái)**.
4. Hiểu mấu chốt: mỗi chữ cái có **2 lựa chọn**, chữ số có **1**, nên số kết quả là **`2^k`** với `k` là số chữ cái; đây là bài **liệt kê tổ hợp** nên độ phức tạp tỉ lệ với kích thước kết quả.
5. Tránh các lỗi kinh điển: gán trực tiếp vào chuỗi (`s[i] = ...` gây `TypeError`), XOR `32` lên **chữ số**, quên rẽ **hai nhánh** cho chữ cái, thêm cùng một `list` vào kết quả mà không `join`/sao chép, và tạo trùng lặp khi "lật" cả chữ số.

## 2. Đề bài & ví dụ

Cho chuỗi `s`. Bạn có thể **đổi từng chữ cái** thành chữ thường hoặc chữ hoa một cách **độc lập** để tạo ra chuỗi khác. Trả về **danh sách mọi chuỗi** có thể tạo ra, theo **thứ tự bất kỳ**.

**Ví dụ 1**

- Input: `s = "a1b2"`
- Output: `["a1b2","a1B2","A1b2","A1B2"]`

**Ví dụ 2**

- Input: `s = "3z4"`
- Output: `["3z4","3Z4"]`

**Ràng buộc:**

- `1 <= s.length <= 12`
- `s` gồm chữ cái thường, chữ cái hoa tiếng Anh và chữ số.

Chú ý: chữ số **không đổi**. Nếu `s` có `k` chữ cái thì có đúng **`2^k`** kết quả, tối đa `2^12 = 4096`. Chữ cái trong `s` có thể đang là hoa hoặc thường; kết quả vẫn gồm mọi tổ hợp.

In [1]:
from typing import List
from itertools import product
import random
import time

examples = [
    ("a1b2", ["a1b2", "a1B2", "A1b2", "A1B2"]),
    ("3z4", ["3z4", "3Z4"]),
]
for s, expected in examples:
    print(repr(s), "-> kỳ vọng", expected, "(sắp xếp:", sorted(expected), ")")

'a1b2' -> kỳ vọng ['a1b2', 'a1B2', 'A1b2', 'A1B2'] (sắp xếp: ['A1B2', 'A1b2', 'a1B2', 'a1b2'] )
'3z4' -> kỳ vọng ['3z4', '3Z4'] (sắp xếp: ['3Z4', '3z4'] )


## 3. Kiến thức nền

### 3.1. Duyệt chuỗi và kiểm tra chữ cái: `str.isalpha`

Chuỗi duyệt được từng ký tự bằng `for ch in s`; `enumerate(s)` cho thêm chỉ số. `ch.isalpha()` trả `True` nếu `ch` là chữ cái, `False` nếu là chữ số hay ký hiệu khác. `ch.isdigit()` kiểm tra chữ số.

In [2]:
s = "a1B2"
for i, ch in enumerate(s):
    print(i, repr(ch), "chữ cái?", ch.isalpha(), "| chữ số?", ch.isdigit())
print("số chữ cái:", sum(ch.isalpha() for ch in s))

0 'a' chữ cái? True | chữ số? False
1 '1' chữ cái? False | chữ số? True
2 'B' chữ cái? True | chữ số? False
3 '2' chữ cái? False | chữ số? True
số chữ cái: 2


### 3.2. Đổi hoa/thường: `lower`, `upper`, `swapcase`

- `ch.lower()` về chữ thường, `ch.upper()` về chữ hoa, `ch.swapcase()` **đảo** (thường ↔ hoa). Với chữ số các hàm này **không làm gì**.

Đây là cách đơn giản nhất, không cần nhớ mã ASCII.

In [3]:
for ch in "aZ7":
    print(repr(ch), "lower:", ch.lower(), "| upper:", ch.upper(), "| swapcase:", ch.swapcase())

'a' lower: a | upper: A | swapcase: A
'Z' lower: z | upper: Z | swapcase: z
'7' lower: 7 | upper: 7 | swapcase: 7


### 3.3. Mã ASCII, `ord`/`chr` và mẹo XOR `32`

`ord(ch)` trả mã số của ký tự; `chr(n)` làm ngược lại. Trong bảng ASCII, chữ thường và chữ hoa **cách nhau đúng `32`**: `ord('a') = 97`, `ord('A') = 65`, `97 - 65 = 32 = 0b100000`. Chữ thường có **bit thứ 5 bằng 1**, chữ hoa có bit đó bằng 0. Vì vậy `ord(ch) ^ 32` (XOR với `32`) **lật bit đó**, đổi thường ↔ hoa. Mẹo này **chỉ đúng với chữ cái**; với chữ số, XOR `32` cho ra ký tự lạ.

In [4]:
print(ord("a"), ord("A"), ord("a") - ord("A"), bin(32))
print(chr(ord("a") ^ 32), chr(ord("Q") ^ 32))        # A q
print("với chữ số '1':", ord("1"), "->", ord("1") ^ 32, repr(chr(ord("1") ^ 32)), "(không phải chữ số!)")

97 65 32 0b100000
A q
với chữ số '1': 49 -> 17 '\x11' (không phải chữ số!)


### 3.4. Chuỗi **bất biến** (immutable): dùng `list` rồi `"".join`

Không gán được `s[0] = "A"` (lỗi `TypeError`) vì chuỗi bất biến. Muốn sửa từng ký tự: đổi sang `list(s)`, sửa phần tử, rồi `"".join(list)` để thành chuỗi.

In [5]:
s = "a1b2"
try:
    s[0] = "A"
except TypeError as e:
    print("TypeError:", e)

t = list(s)
t[0] = "A"
print(t, "->", "".join(t))

TypeError: 'str' object does not support item assignment
['A', '1', 'b', '2'] -> A1b2


### 3.5. Đệ quy quay lui: chọn → đi tiếp → bỏ chọn

Với bài liệt kê, ta xử lý từng vị trí một: ở mỗi chữ cái **rẽ hai nhánh** (giữ nguyên / đổi), ở chữ số chỉ một nhánh. Khi đến cuối chuỗi, ta có một kết quả. Sau khi quay lại từ nhánh "đổi", cần **đổi lại** để trả trạng thái như cũ (bỏ chọn). Ví dụ nhỏ: liệt kê mọi chuỗi nhị phân độ dài 3.

In [6]:
def binary_strings(n):
    out, cur = [], []
    def dfs(i):
        if i == n:
            out.append("".join(cur))
            return
        for bit in "01":
            cur.append(bit)       # chọn
            dfs(i + 1)            # đi tiếp
            cur.pop()             # bỏ chọn
    dfs(0)
    return out

print(binary_strings(3))

['000', '001', '010', '011', '100', '101', '110', '111']


### 3.6. Thao tác bit: `1 << k`, `&`, `>>`, `bit_length`

Một số nguyên `mask` có thể mã hóa một **tập lựa chọn**: bit thứ `i` bằng 1 nghĩa là "lật chữ cái thứ `i`".

- `1 << i`: số có đúng bit `i` bật (`1 << 3 = 8`).
- `mask >> i & 1`: lấy bit thứ `i` (0 hoặc 1).
- `range(1 << k)` duyệt mọi `2^k` mặt nạ từ `0` đến `2^k - 1`.
- `(x & -x)` giữ lại **bit thấp nhất đang bật** của `x`; `.bit_length() - 1` cho **vị trí** của nó (dùng ở Cách 5).

In [7]:
k = 3
for mask in range(1 << k):
    bits = [mask >> i & 1 for i in range(k)]
    print(f"mask={mask} ({mask:03b}) -> bit0..bit2 = {bits}")
for x in (1, 2, 3, 4, 6, 8):
    low = x & -x
    print(f"x={x}: bit thấp nhất bật = {low}, vị trí = {low.bit_length() - 1}")

mask=0 (000) -> bit0..bit2 = [0, 0, 0]
mask=1 (001) -> bit0..bit2 = [1, 0, 0]
mask=2 (010) -> bit0..bit2 = [0, 1, 0]
mask=3 (011) -> bit0..bit2 = [1, 1, 0]
mask=4 (100) -> bit0..bit2 = [0, 0, 1]
mask=5 (101) -> bit0..bit2 = [1, 0, 1]
mask=6 (110) -> bit0..bit2 = [0, 1, 1]
mask=7 (111) -> bit0..bit2 = [1, 1, 1]
x=1: bit thấp nhất bật = 1, vị trí = 0
x=2: bit thấp nhất bật = 2, vị trí = 1
x=3: bit thấp nhất bật = 1, vị trí = 0
x=4: bit thấp nhất bật = 4, vị trí = 2
x=6: bit thấp nhất bật = 2, vị trí = 1
x=8: bit thấp nhất bật = 8, vị trí = 3


### 3.7. `itertools.product`

`product(A, B, C)` cho **tích Descartes**: mọi bộ `(a, b, c)` với `a ∈ A`, `b ∈ B`, `c ∈ C`. Mỗi vị trí cho một danh sách lựa chọn; `product(*lists)` ghép mọi lựa chọn. Rất hợp bài này (chữ cái: 2 lựa chọn; chữ số: 1 lựa chọn), và ta dùng làm **oracle** để kiểm tra.

In [8]:
choices = [("a", "A"), ("1",), ("b", "B"), ("2",)]
print([ "".join(p) for p in product(*choices) ])

['a1b2', 'a1B2', 'A1b2', 'A1B2']


## 4. Ý tưởng nền tảng / trực giác

Tính tay với `s = "a1b2"`. Có `4` ký tự, trong đó `2` chữ cái (`a` ở vị trí 0 và `b` ở vị trí 2).

- Vị trí 0 (`a`): 2 lựa chọn `a` hoặc `A`.
- Vị trí 1 (`1`): 1 lựa chọn `1`.
- Vị trí 2 (`b`): 2 lựa chọn `b` hoặc `B`.
- Vị trí 3 (`2`): 1 lựa chọn `2`.

Số kết quả: `2 * 1 * 2 * 1 = 4 = 2^2`. Vẽ cây quyết định:

```
              ""
         /          \
       a             A
       |             |
      a1            A1
     /   \         /   \
  a1b    a1B    A1b    A1B
   |      |      |      |
 a1b2   a1B2   A1b2   A1B2
```

Mỗi đường từ gốc tới lá là một chuỗi kết quả. Chữ số không tạo nhánh mới.

**Trực giác:** đây giống như có `k` công tắc (một công tắc cho mỗi chữ cái), mỗi công tắc bật/tắt (hoa/thường). Mỗi trạng thái của `k` công tắc cho đúng một chuỗi: có `2^k` chuỗi.

In [9]:
s = "a1b2"
letters = [i for i, ch in enumerate(s) if ch.isalpha()]
k = len(letters)
print("vị trí các chữ cái:", letters, "| k =", k, "| số kết quả =", 2 ** k)

# làm tay bằng product
choices = [(ch.lower(), ch.upper()) if ch.isalpha() else (ch,) for ch in s]
print(choices)
print(sorted("".join(p) for p in product(*choices)))

vị trí các chữ cái: [0, 2] | k = 2 | số kết quả = 4
[('a', 'A'), ('1',), ('b', 'B'), ('2',)]
['A1B2', 'A1b2', 'a1B2', 'a1b2']


## 5. Cách 1: Brute Force (thử mọi mặt nạ cho mọi ký tự, rồi khử trùng)

**Ý tưởng "thô":** coi **mọi ký tự** (kể cả chữ số) như có một công tắc "lật". Với mỗi mặt nạ `mask` từ `0` đến `2^n - 1` (`n = len(s)`), lật những ký tự có bit tương ứng bằng 1 (dùng `swapcase`, chữ số không đổi). Vì lật chữ số **không thay đổi gì**, nhiều mặt nạ cho cùng một chuỗi, nên ta dùng `set` để **khử trùng**.

In [10]:
def brute_force(s):
    n = len(s)
    seen = set()
    for mask in range(1 << n):                      # mọi tổ hợp lật trên mọi ký tự
        t = []
        for i, ch in enumerate(s):
            t.append(ch.swapcase() if mask >> i & 1 else ch)
        seen.add("".join(t))
    return list(seen)

for s, expected in examples:
    got = brute_force(s)
    print(repr(s), "->", sorted(got))
    assert sorted(got) == sorted(expected)

'a1b2' -> ['A1B2', 'A1b2', 'a1B2', 'a1b2']
'3z4' -> ['3Z4', '3z4']


**Độ phức tạp:** `2^n` mặt nạ, mỗi mặt nạ dựng chuỗi `O(n)`: **Time `O(n * 2^n)`**, nhưng chỉ có `2^k` kết quả thật (`k <= n`). Space `O(2^k * n)` cho tập kết quả.

**Nút thắt:** khi `s` nhiều chữ số, ta làm việc dư rất nhiều (ví dụ `"1111"` có 16 mặt nạ nhưng 1 kết quả). Cải tiến: chỉ để **công tắc cho chữ cái**, bỏ qua chữ số, nhờ đó không trùng lặp và không cần `set`.

In [11]:
s = "1111111111a"
t0 = time.perf_counter()
r = brute_force(s)
print(f"'{s}': {len(r)} kết quả, nhưng đã thử {2 ** len(s)} mặt nạ ({(time.perf_counter() - t0) * 1000:.1f} ms)")

'1111111111a': 2 kết quả, nhưng đã thử 2048 mặt nạ (2.2 ms)


## 6. Ý tưởng thuật toán / bước đột phá

Chữ số **không có lựa chọn**, nên chỉ chữ cái mới "phân nhánh". Nếu ta chỉ rẽ nhánh ở chữ cái thì **mỗi đường đi là một kết quả riêng biệt**, không có trùng, và số kết quả đúng `2^k`.

Từ đó có nhiều cách:

- **Quay lui (DFS):** đi từ trái sang phải; ở chữ cái rẽ hai nhánh, ở chữ số đi thẳng; đến cuối thì ghi kết quả.
- **Mở rộng dần (BFS):** giữ danh sách các tiền tố; với mỗi ký tự, mỗi tiền tố sinh thêm 1 (chữ số) hoặc 2 (chữ cái) tiền tố mới.
- **Mặt nạ bit trên các chữ cái:** gán mỗi chữ cái một bit; duyệt `0 .. 2^k - 1`.
- **Mã Gray:** duyệt các mặt nạ theo thứ tự mà hai mặt nạ liên tiếp chỉ khác **một bit**, nên mỗi bước chỉ **lật đúng một chữ cái**.

Mọi cách đều tạo đúng `2^k` chuỗi; độ phức tạp thời gian ít nhất là `O(n * 2^k)` vì phải **xuất** từng chuỗi dài `n`.

In [12]:
def count_results(s):
    return 2 ** sum(ch.isalpha() for ch in s)

for s in ["a1b2", "3z4", "12345", "abc", "aBc", "ab12cd34"]:
    print(f"{s:10s} -> {count_results(s):3d} kết quả (2^{sum(ch.isalpha() for ch in s)})")

a1b2       ->   4 kết quả (2^2)
3z4        ->   2 kết quả (2^1)
12345      ->   1 kết quả (2^0)
abc        ->   8 kết quả (2^3)
aBc        ->   8 kết quả (2^3)
ab12cd34   ->  16 kết quả (2^4)


## 7. Các cách tiếp theo

Tổng cộng **5 cách**: Cách 1 mặt nạ trên mọi ký tự + `set` (đã làm); Cách 2 DFS quay lui; Cách 3 mở rộng dần danh sách (BFS); Cách 4 mặt nạ bit chỉ trên chữ cái; Cách 5 mã Gray.

### 7.1. Cách 2: DFS quay lui (cách chính)

**Giải thích từng đoạn:**

1. `t = list(s)`: bản sao có thể sửa; `ans` chứa kết quả.
2. `dfs(i)`: xét vị trí `i`. Nếu `i == len(t)`: đã quyết định xong mọi ký tự → thêm `"".join(t)` vào `ans`.
3. Ở mỗi `i`: **đầu tiên** đi tiếp `dfs(i + 1)` với ký tự **giữ nguyên**.
4. Nếu `t[i]` là chữ cái: **đổi hoa/thường** bằng XOR `32` rồi `dfs(i + 1)` lần nữa (nhánh thứ hai). Hai nhánh này khác nhau nên không trùng. (Ta không cần đổi lại: nhánh "giữ nguyên" đã chạy xong **trước**, nên sau nhánh "đổi" không còn ai dùng `t[i]` ở trạng thái cũ.)

In [13]:
def dfs_backtrack(s):
    t = list(s)
    ans = []

    def dfs(i):
        if i == len(t):
            ans.append("".join(t))
            return
        dfs(i + 1)                                   # nhánh 1: giữ nguyên
        if t[i].isalpha():
            t[i] = chr(ord(t[i]) ^ 32)               # đổi hoa/thường
            dfs(i + 1)                               # nhánh 2: đã đổi
    dfs(0)
    return ans

for s, expected in examples:
    got = dfs_backtrack(s)
    print(repr(s), "->", got)
    assert sorted(got) == sorted(expected)

'a1b2' -> ['a1b2', 'a1B2', 'A1B2', 'A1b2']
'3z4' -> ['3z4', '3Z4']


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def letterCasePermutation(self, s: str) -> List[str]:
        def dfs(i: int) -> None:
            if i >= len(t):
                ans.append("".join(t))
                return
            dfs(i + 1)
            if t[i].isalpha():
                t[i] = chr(ord(t[i]) ^ 32)
                dfs(i + 1)

        t = list(s)
        ans = []
        dfs(0)
        return ans
```

**Độ phức tạp:** mỗi lá cho một chuỗi `O(n)`; có `2^k` lá nên Time **`O(n * 2^k)`** (với `k <= n`, nhớ `n <= 12`). Space `O(n)` cho đệ quy và `t` (không tính danh sách kết quả).

### 7.2. Cách 3: Mở rộng dần danh sách (BFS)

**Ý tưởng:** bắt đầu từ danh sách `[""]`. Với mỗi ký tự `ch` của `s`: nếu là **chữ số** thì nối `ch` vào **mọi** chuỗi hiện có; nếu là **chữ cái** thì nối cả `ch.lower()` **và** `ch.upper()` vào mọi chuỗi (danh sách **gấp đôi** kích thước). Sau khi xử lý hết, danh sách chính là đáp án.

**Giải thích từng đoạn:**

1. `res = [""]`.
2. Với mỗi `ch`: `res = [p + c for p in res for c in options]`, trong đó `options = (ch.lower(), ch.upper())` nếu chữ cái, ngược lại `(ch,)`.
3. Cuối cùng trả `res`.

In [14]:
def bfs_expand(s):
    res = [""]
    for ch in s:
        options = (ch.lower(), ch.upper()) if ch.isalpha() else (ch,)
        res = [p + c for p in res for c in options]
    return res

for s, expected in examples:
    got = bfs_expand(s)
    print(repr(s), "->", got)
    assert sorted(got) == sorted(expected)

'a1b2' -> ['a1b2', 'a1B2', 'A1b2', 'A1B2']
'3z4' -> ['3z4', '3Z4']


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def letterCasePermutation(self, s: str) -> List[str]:
        res = [""]
        for ch in s:
            options = (ch.lower(), ch.upper()) if ch.isalpha() else (ch,)
            res = [p + c for p in res for c in options]
        return res
```

**Độ phức tạp:** ở bước `i`, danh sách có `2^(số chữ cái đã gặp)` chuỗi dài `i`. Tổng công việc `O(n * 2^k)` (hình học, bị chi phối bởi bước cuối). Space `O(n * 2^k)` cho danh sách trung gian (gấp khoảng 2 lần kết quả khi tạo danh sách mới).

### 7.3. Cách 4: Mặt nạ bit chỉ trên các chữ cái

**Ý tưởng:** gom **vị trí các chữ cái** vào list `pos` (độ dài `k`). Với mỗi `mask` từ `0` đến `2^k - 1`: bit thứ `j` của `mask` quyết định chữ cái ở vị trí `pos[j]` là **thường** (0) hay **hoa** (1). Không có chữ số nào bị "lật" nên **không trùng**.

**Giải thích từng đoạn:**

1. `pos = [i for i, ch in enumerate(s) if ch.isalpha()]`.
2. Với mỗi `mask`: sao chép `t = list(s)`; với `j, p in enumerate(pos)`: `t[p] = s[p].upper()` nếu `mask >> j & 1` ngược lại `s[p].lower()`.
3. `"".join(t)` vào kết quả.

In [15]:
def bitmask_letters(s):
    pos = [i for i, ch in enumerate(s) if ch.isalpha()]
    k = len(pos)
    ans = []
    for mask in range(1 << k):
        t = list(s)
        for j, p in enumerate(pos):
            t[p] = s[p].upper() if mask >> j & 1 else s[p].lower()
        ans.append("".join(t))
    return ans

for s, expected in examples:
    got = bitmask_letters(s)
    print(repr(s), "->", got)
    assert sorted(got) == sorted(expected)

'a1b2' -> ['a1b2', 'A1b2', 'a1B2', 'A1B2']
'3z4' -> ['3z4', '3Z4']


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def letterCasePermutation(self, s: str) -> List[str]:
        pos = [i for i, ch in enumerate(s) if ch.isalpha()]
        ans = []
        for mask in range(1 << len(pos)):
            t = list(s)
            for j, p in enumerate(pos):
                t[p] = s[p].upper() if mask >> j & 1 else s[p].lower()
            ans.append("".join(t))
        return ans
```

**Độ phức tạp:** `2^k` mặt nạ, mỗi mặt nạ `O(n)`: Time `O(n * 2^k)`, Space `O(n)` (không tính kết quả). Không đệ quy, không `set`.

### 7.4. Cách 5: Mã Gray (mỗi bước chỉ lật một chữ cái)

**Ý tưởng khác:** trong Cách 4, mỗi chuỗi được dựng lại từ đầu (`O(n)`). Nếu duyệt các mặt nạ theo thứ tự **mã Gray**, hai mặt nạ liên tiếp chỉ khác **đúng một bit**: ta chỉ cần **lật một chữ cái** của chuỗi hiện tại (cập nhật `O(1)`), rồi nối `join` để lưu (`O(n)`).

**Cách tìm chữ cái cần lật:** ở bước `i` (từ `1` đến `2^k - 1`), vị trí bit cần lật là **vị trí bit thấp nhất đang bật** của `i`: `(i & -i).bit_length() - 1`. Dãy vị trí lật khi `k = 3`: `0, 1, 0, 2, 0, 1, 0` (tính chất "ruler sequence" của mã Gray).

**Giải thích từng đoạn:**

1. `t = list(s)`, `ans = ["".join(t)]` (chuỗi đầu: nguyên bản).
2. Với `i` từ `1` đến `2^k - 1`: `j = (i & -i).bit_length() - 1`, lật `t[pos[j]]` bằng XOR `32`, rồi thêm `"".join(t)`.
3. Qua `2^k` trạng thái, mọi tổ hợp hoa/thường được duyệt đúng một lần.

In [16]:
def gray_code(s):
    pos = [i for i, ch in enumerate(s) if ch.isalpha()]
    k = len(pos)
    t = list(s)
    ans = ["".join(t)]
    for i in range(1, 1 << k):
        j = (i & -i).bit_length() - 1              # chữ cái cần lật ở bước này
        p = pos[j]
        t[p] = chr(ord(t[p]) ^ 32)                 # lật đúng một chữ cái
        ans.append("".join(t))
    return ans

for s, expected in examples:
    got = gray_code(s)
    print(repr(s), "->", got)
    assert sorted(got) == sorted(expected)
print("dãy chỉ số bit lật với k = 3:", [(i & -i).bit_length() - 1 for i in range(1, 8)])

'a1b2' -> ['a1b2', 'A1b2', 'A1B2', 'a1B2']
'3z4' -> ['3z4', '3Z4']
dãy chỉ số bit lật với k = 3: [0, 1, 0, 2, 0, 1, 0]


**Code nộp LeetCode (Cách 5):**

```python
class Solution:
    def letterCasePermutation(self, s: str) -> List[str]:
        pos = [i for i, ch in enumerate(s) if ch.isalpha()]
        t = list(s)
        ans = ["".join(t)]
        for i in range(1, 1 << len(pos)):
            p = pos[(i & -i).bit_length() - 1]
            t[p] = chr(ord(t[p]) ^ 32)
            ans.append("".join(t))
        return ans
```

**Độ phức tạp:** `2^k` bước, mỗi bước lật `O(1)` rồi `join` `O(n)` để lưu: Time `O(n * 2^k)` (phần "tính chuỗi tiếp theo" chỉ `O(1)`; nếu chỉ cần duyệt không lưu thì `O(2^k)`). Space `O(n)` (không tính kết quả). Cách này hữu ích khi bạn **chỉ duyệt** các chuỗi một lần mà không cần giữ.

## 8. TRACE từng bước

### 8.1. Trace DFS với `s = "a1b2"`

Quy ước: `t` là list ký tự hiện tại; `dfs(i)` xét vị trí `i`.

| Lời gọi | `t` | Quyết định | Kết quả |
|---|---|---|---|
| `dfs(0)` | `a1b2` | gọi `dfs(1)` (giữ `a`) | |
| `dfs(1)` | `a1b2` | `1` là chữ số, chỉ `dfs(2)` | |
| `dfs(2)` | `a1b2` | gọi `dfs(3)` (giữ `b`) | |
| `dfs(3)` | `a1b2` | `2` là chữ số: `dfs(4)` | |
| `dfs(4)` | `a1b2` | `i == 4`: ghi | **`a1b2`** |
| (quay về `dfs(2)`) | `a1b2` | `b` là chữ cái: đổi → `a1B2`, gọi `dfs(3)` | |
| ... `dfs(4)` | `a1B2` | ghi | **`a1B2`** |
| (quay về `dfs(0)`) | `a1B2` | `a` là chữ cái: đổi → `A1B2`; đi tiếp: `dfs(1)`... | ghi `A1B2`, rồi đổi `B` lại ra `A1b2` |

Thứ tự thật khi chạy: `a1b2, a1B2, A1B2, A1b2`.

In [17]:
def trace_dfs(s):
    t = list(s)
    ans = []
    def dfs(i):
        indent = "    " * i
        if i == len(t):
            ans.append("".join(t))
            print(f"{indent}i={i}: hết chuỗi -> ghi {ans[-1]!r}")
            return
        print(f"{indent}i={i}: t={''.join(t)!r}, ký tự {t[i]!r}, nhánh 'giữ nguyên'")
        dfs(i + 1)
        if t[i].isalpha():
            old = t[i]
            t[i] = chr(ord(t[i]) ^ 32)
            print(f"{indent}i={i}: đổi {old!r} -> {t[i]!r}, t={''.join(t)!r}, nhánh 'đã đổi'")
            dfs(i + 1)
    dfs(0)
    return ans

res = trace_dfs("a1b2")
print("kết quả theo thứ tự sinh:", res)
assert sorted(res) == sorted(examples[0][1])

i=0: t='a1b2', ký tự 'a', nhánh 'giữ nguyên'
    i=1: t='a1b2', ký tự '1', nhánh 'giữ nguyên'
        i=2: t='a1b2', ký tự 'b', nhánh 'giữ nguyên'
            i=3: t='a1b2', ký tự '2', nhánh 'giữ nguyên'
                i=4: hết chuỗi -> ghi 'a1b2'
        i=2: đổi 'b' -> 'B', t='a1B2', nhánh 'đã đổi'
            i=3: t='a1B2', ký tự '2', nhánh 'giữ nguyên'
                i=4: hết chuỗi -> ghi 'a1B2'
i=0: đổi 'a' -> 'A', t='A1B2', nhánh 'đã đổi'
    i=1: t='A1B2', ký tự '1', nhánh 'giữ nguyên'
        i=2: t='A1B2', ký tự 'B', nhánh 'giữ nguyên'
            i=3: t='A1B2', ký tự '2', nhánh 'giữ nguyên'
                i=4: hết chuỗi -> ghi 'A1B2'
        i=2: đổi 'B' -> 'b', t='A1b2', nhánh 'đã đổi'
            i=3: t='A1b2', ký tự '2', nhánh 'giữ nguyên'
                i=4: hết chuỗi -> ghi 'A1b2'
kết quả theo thứ tự sinh: ['a1b2', 'a1B2', 'A1B2', 'A1b2']


Quan sát: nhánh "đổi" **không đổi lại** `t[i]` sau khi xong (vì không còn nhánh nào ở vị trí đó); nhưng cha của nó (`dfs(i - 1)`) khi chuyển sang nhánh "đổi" của chính nó vẫn thấy `t` ở đúng trạng thái mong muốn, vì nhánh "giữ nguyên" của `dfs(i)` luôn chạy **trước**. Đây là lý do thứ tự kết quả là `a1b2, a1B2, A1B2, A1b2` (đảo chiều ở nhánh thứ hai). Với bài này thứ tự không quan trọng.

### 8.2. Trace Cách 3 (mở rộng danh sách) với `s = "a1b2"`

| Ký tự | Lựa chọn | Danh sách sau bước |
|---|---|---|
| (đầu) | | `['']` |
| `a` | `a`, `A` | `['a', 'A']` |
| `1` | `1` | `['a1', 'A1']` |
| `b` | `b`, `B` | `['a1b', 'a1B', 'A1b', 'A1B']` |
| `2` | `2` | `['a1b2', 'a1B2', 'A1b2', 'A1B2']` |

In [18]:
def trace_bfs(s):
    res = [""]
    print("bắt đầu:", res)
    for ch in s:
        options = (ch.lower(), ch.upper()) if ch.isalpha() else (ch,)
        res = [p + c for p in res for c in options]
        print(f"ký tự {ch!r} lựa chọn {options}: {res}")
    return res

assert trace_bfs("a1b2") == ["a1b2", "a1B2", "A1b2", "A1B2"]

bắt đầu: ['']
ký tự 'a' lựa chọn ('a', 'A'): ['a', 'A']
ký tự '1' lựa chọn ('1',): ['a1', 'A1']
ký tự 'b' lựa chọn ('b', 'B'): ['a1b', 'a1B', 'A1b', 'A1B']
ký tự '2' lựa chọn ('2',): ['a1b2', 'a1B2', 'A1b2', 'A1B2']


### 8.3. Trace mã Gray với `s = "a1b2"` (`k = 2`, vị trí chữ cái `[0, 2]`)

| `i` | `(i & -i)` | bit lật `j` | chữ cái lật (`pos[j]`) | chuỗi sau bước |
|---|---|---|---|---|
| (đầu) | | | | `a1b2` |
| 1 | 1 | 0 | vị trí 0 (`a`→`A`) | `A1b2` |
| 2 | 2 | 1 | vị trí 2 (`b`→`B`) | `A1B2` |
| 3 | 1 | 0 | vị trí 0 (`A`→`a`) | `a1B2` |

Đủ `4 = 2^2` chuỗi: `a1b2, A1b2, A1B2, a1B2`; hai chuỗi liên tiếp khác nhau đúng một chữ cái.

In [19]:
def trace_gray(s):
    pos = [i for i, ch in enumerate(s) if ch.isalpha()]
    k = len(pos)
    t = list(s)
    print(f"đầu: {''.join(t)!r} | vị trí chữ cái {pos}")
    out = ["".join(t)]
    for i in range(1, 1 << k):
        low = i & -i
        j = low.bit_length() - 1
        p = pos[j]
        old = t[p]
        t[p] = chr(ord(t[p]) ^ 32)
        out.append("".join(t))
        print(f"i={i}: i&-i={low}, j={j} -> lật vị trí {p}: {old!r}->{t[p]!r}, chuỗi {out[-1]!r}")
    return out

res = trace_gray("a1b2")
assert res == ["a1b2", "A1b2", "A1B2", "a1B2"]
# hai chuỗi liên tiếp khác nhau đúng một ký tự
assert all(sum(x != y for x, y in zip(a, b)) == 1 for a, b in zip(res, res[1:]))

đầu: 'a1b2' | vị trí chữ cái [0, 2]
i=1: i&-i=1, j=0 -> lật vị trí 0: 'a'->'A', chuỗi 'A1b2'
i=2: i&-i=2, j=1 -> lật vị trí 2: 'b'->'B', chuỗi 'A1B2'
i=3: i&-i=1, j=0 -> lật vị trí 0: 'A'->'a', chuỗi 'a1B2'


## 9. Code nộp LeetCode

Ta nộp **Cách 2 (DFS quay lui)**: ngắn gọn và là cách kinh điển cho bài liệt kê.

In [20]:
class Solution:
    def letterCasePermutation(self, s: str) -> List[str]:
        def dfs(i: int) -> None:
            if i >= len(t):
                ans.append("".join(t))          # đủ n ký tự: ghi một kết quả
                return
            dfs(i + 1)                           # nhánh giữ nguyên ký tự i
            if t[i].isalpha():
                t[i] = chr(ord(t[i]) ^ 32)       # đổi hoa/thường (chỉ cho chữ cái)
                dfs(i + 1)                       # nhánh đã đổi

        t = list(s)
        ans = []
        dfs(0)
        return ans


sol = Solution()
for s, expected in examples:
    got = sol.letterCasePermutation(s)
    print(repr(s), "->", got, "| sắp xếp khớp kỳ vọng:", sorted(got) == sorted(expected))
    assert sorted(got) == sorted(expected)

'a1b2' -> ['a1b2', 'a1B2', 'A1B2', 'A1b2'] | sắp xếp khớp kỳ vọng: True
'3z4' -> ['3z4', '3Z4'] | sắp xếp khớp kỳ vọng: True


## 10. Vì sao đúng

**Mô hình.** Chuỗi kết quả `r` hợp lệ ⇔ `len(r) == len(s)`, và với mỗi vị trí `i`: nếu `s[i]` là chữ số thì `r[i] == s[i]`; nếu là chữ cái thì `r[i]` là `s[i].lower()` hoặc `s[i].upper()`. Số chuỗi hợp lệ là `2^k`.

**DFS sinh đúng và đủ.** Quy nạp theo vị trí `i`: giả sử `dfs(i)` được gọi với `t[0..i-1]` đã được quyết định (một trong các lựa chọn hợp lệ cho các vị trí đầu) và `t[i..]` còn nguyên bản. Nếu `s[i]` là chữ số, chỉ có một lựa chọn, `dfs(i+1)` tiếp. Nếu là chữ cái, nhánh đầu giữ ký tự gốc, nhánh sau đổi ký tự; hai nhánh phủ đủ `{lower, upper}` (vì ký tự gốc là một trong hai, nhánh đổi là cái còn lại). Khi `i == n`, `t` là một chuỗi hợp lệ. Mỗi lá tương ứng một dãy lựa chọn khác nhau ⇒ chuỗi khác nhau (khác ở ít nhất một chữ cái) ⇒ **không trùng**.

**Lưu ý trạng thái `t`.** Mỗi lần `dfs(i)` được gọi, ký tự `t[i]` đang ở **một trong hai dạng** (dạng nào cũng được). `dfs(i)` sinh kết quả cho dạng hiện tại, rồi **lật** `t[i]` và sinh tiếp cho dạng còn lại; nên qua một lần gọi, **cả hai dạng** của ký tự `i` đều được duyệt đúng một lần (dù dạng bắt đầu là gì). Hệ quả: khi `dfs(i)` trở về, `t[i]` đã bị lật so với lúc vào, nhưng điều đó không ảnh hưởng tính đúng, chỉ làm thứ tự sinh kết quả khác đi (ví dụ `a1b2, a1B2, A1B2, A1b2`). Vì đề cho phép thứ tự bất kỳ nên không sao.

**Cách khác.** Cách 3 (mở rộng) đúng vì mỗi bước nhân tập tiền tố với tập lựa chọn của ký tự. Cách 4 đúng vì mỗi `mask` là một ánh xạ 1-1 từ tập công tắc sang chuỗi. Cách 5 đúng vì dãy Gray đi qua mọi mặt nạ đúng một lần (mỗi bước lật một bit).

**Kiểm chứng bằng code:** hàm kiểm tra tính hợp lệ (đúng `2^k` chuỗi, khác nhau, đúng dạng) cho mọi cách, và đối chiếu tập kết quả với `itertools.product` (oracle).

In [21]:
def is_valid_answer(s, result):
    """Kiểm tra tính chất, không cần so đáp án cứng (thứ tự bất kỳ)."""
    k = sum(ch.isalpha() for ch in s)
    if len(result) != 2 ** k:
        return False
    if len(set(result)) != len(result):               # không trùng
        return False
    for r in result:
        if len(r) != len(s):
            return False
        for a, b in zip(s, r):
            if a.isdigit():
                if a != b:
                    return False
            elif b.lower() != a.lower() or not b.isalpha():
                return False
    return True

def oracle(s):
    choices = [(ch.lower(), ch.upper()) if ch.isalpha() else (ch,) for ch in s]
    return sorted("".join(p) for p in product(*choices))

for s, expected in examples:
    assert oracle(s) == sorted(expected)
    assert is_valid_answer(s, expected)
print("oracle và bộ kiểm tra khớp 2 ví dụ")

rng = random.Random(784)
alphabet = "abcXYZ019"
for _ in range(300):
    s = "".join(rng.choice(alphabet) for _ in range(rng.randint(1, 10)))
    res = Solution().letterCasePermutation(s)
    assert is_valid_answer(s, res)
    assert sorted(res) == oracle(s)
print("300 chuỗi ngẫu nhiên: kết quả hợp lệ và khớp oracle")

oracle và bộ kiểm tra khớp 2 ví dụ
300 chuỗi ngẫu nhiên: kết quả hợp lệ và khớp oracle


## 11. Time/Space Complexity

Gọi `n = len(s)`, `k` là số chữ cái (`k <= n`). Có đúng `2^k` kết quả, mỗi kết quả dài `n`.

| Cách | Time | Space (không tính kết quả) |
|---|---|---|
| 1. Mặt nạ trên mọi ký tự + `set` | `O(n * 2^n)` | `O(n * 2^k)` (set) |
| 2. DFS quay lui | `O(n * 2^k)` | `O(n)` |
| 3. Mở rộng dần (BFS) | `O(n * 2^k)` | `O(n * 2^k)` (danh sách trung gian) |
| 4. Mặt nạ bit trên chữ cái | `O(n * 2^k)` | `O(n)` |
| 5. Mã Gray | `O(n * 2^k)` (chỉ `O(2^k)` nếu không cần lưu chuỗi) | `O(n)` |

**Tại sao không thể tốt hơn:** chỉ riêng việc **xuất** `2^k` chuỗi mỗi chuỗi `n` ký tự đã tốn `Ω(n * 2^k)`. Với `n <= 12`, tối đa `4096 * 12 ≈ 49.000` ký tự, rất nhỏ.

Chú ý: Cách 1 tốn `2^n` (không phải `2^k`) vì thử cả công tắc cho chữ số: nên chậm khi nhiều chữ số.

## 12. Bộ test

Vì **thứ tự kết quả không quan trọng**, mọi test so sánh `sorted(kết_quả)` với đáp án đã `sorted` (hoặc dùng bộ kiểm tra tính chất). Nhóm test: ví dụ đề; edge case (toàn chữ số, một ký tự, chữ cái hoa trong input, toàn chữ cái dài `12`); phản ví dụ cho các cách làm sai; đối chiếu **5 cách + lớp nộp** với **oracle `itertools.product`** trên chuỗi ngẫu nhiên có seed; test kích thước lớn nhất (`n = 12`, `4096` kết quả).

In [22]:
methods = {
    "brute_force": brute_force,
    "dfs_backtrack": dfs_backtrack,
    "bfs_expand": bfs_expand,
    "bitmask_letters": bitmask_letters,
    "gray_code": gray_code,
    "Solution": Solution().letterCasePermutation,
}

# 12.1 ví dụ của đề (so sánh sau khi sort)
for s, expected in examples:
    for name, fn in methods.items():
        assert sorted(fn(s)) == sorted(expected), name
print("2 ví dụ: cả", len(methods), "cách đúng")

# 12.2 edge case
edge = {
    "7": ["7"],                                  # một chữ số
    "a": ["A", "a"],                             # một chữ cái
    "Z": ["Z", "z"],                             # chữ hoa ở input cũng cho cả hai dạng
    "12345": ["12345"],                          # toàn chữ số: đúng một kết quả
    "ab": ["AB", "Ab", "aB", "ab"],              # toàn chữ cái
    "A1b": ["A1B", "A1b", "a1B", "a1b"],         # input lẫn hoa/thường: kết quả như nhau
    "0a0": ["0A0", "0a0"],
}
for s, expected in edge.items():
    for name, fn in methods.items():
        assert sorted(fn(s)) == sorted(expected), (name, s, fn(s))
    assert is_valid_answer(s, expected)
print("edge case: OK")

# kết quả không phụ thuộc hoa/thường của input
for s in ["a1b2", "A1B2", "a1B2", "A1b2"]:
    for name, fn in methods.items():
        assert sorted(fn(s)) == sorted(examples[0][1])
print("kết quả không phụ thuộc kiểu chữ ban đầu: OK")

2 ví dụ: cả 6 cách đúng
edge case: OK
kết quả không phụ thuộc kiểu chữ ban đầu: OK


In [23]:
# 12.3 phản ví dụ: các cách làm SAI phổ biến
def wrong_xor_digit(s):                          # XOR 32 cả lên chữ số
    t = list(s)
    ans = []
    def dfs(i):
        if i == len(t):
            ans.append("".join(t))
            return
        dfs(i + 1)
        t[i] = chr(ord(t[i]) ^ 32)               # SAI: không kiểm tra isalpha
        dfs(i + 1)
    dfs(0)
    return ans

def wrong_no_branch(s):                          # chỉ đổi thành hoa, không rẽ hai nhánh
    return ["".join(c.upper() if c.isalpha() else c for c in s)]

def wrong_alias(s):                              # thêm chính list t vào kết quả (không join / sao chép)
    t = list(s)
    ans = []
    def dfs(i):
        if i == len(t):
            ans.append(t)                        # SAI: cùng một đối tượng
            return
        dfs(i + 1)
        if t[i].isalpha():
            t[i] = chr(ord(t[i]) ^ 32)
            dfs(i + 1)
    dfs(0)
    return ["".join(x) for x in ans]

def wrong_dup(s):                                # mặt nạ trên mọi ký tự nhưng quên khử trùng
    out = []
    for mask in range(1 << len(s)):
        out.append("".join(c.swapcase() if mask >> i & 1 else c for i, c in enumerate(s)))
    return out

def wrong_only_from_lower(s):                    # giả định input toàn thường: chỉ mặt nạ lật giữa 'a' và 'A' theo upper()
    pos = [i for i, c in enumerate(s) if c.isalpha()]
    out = []
    for mask in range(1 << len(pos)):
        t = list(s)
        for j, p in enumerate(pos):
            if mask >> j & 1:
                t[p] = t[p].upper()              # SAI nếu input đã là hoa: không bao giờ tạo chữ thường
        out.append("".join(t))
    return out

s = "a1b"
print("đúng               :", sorted(Solution().letterCasePermutation(s)))
print("XOR cả chữ số      :", len(wrong_xor_digit(s)), "chuỗi, ví dụ", repr(wrong_xor_digit(s)[1]))
print("không rẽ hai nhánh :", wrong_no_branch(s))
print("alias list         :", wrong_alias(s))
print("không khử trùng    :", len(wrong_dup("a1b")), "chuỗi (đúng 4)")
print("input hoa 'A1B'    :", sorted(set(wrong_only_from_lower("A1B"))), "(thiếu dạng thường)")
assert len(wrong_xor_digit(s)) != 4 or any(not x.replace("a", "").replace("A", "").replace("b", "").replace("B", "").isdigit() for x in wrong_xor_digit(s))
assert wrong_no_branch(s) != sorted(Solution().letterCasePermutation(s))
assert len(wrong_dup(s)) != 4
assert sorted(set(wrong_only_from_lower("A1B"))) != oracle("A1B")
print("các cách sai bị phát hiện")

đúng               :

 ['A1B', 'A1b', 'a1B', 'a1b']
XOR cả chữ số      : 8 chuỗi, ví dụ 'a1B'
không rẽ hai nhánh : ['A1B']
alias list         : ['A1b', 'A1b', 'A1b', 'A1b']
không khử trùng    : 8 chuỗi (đúng 4)
input hoa 'A1B'    : ['A1B'] (thiếu dạng thường)
các cách sai bị phát hiện


In [24]:
# 12.4 đối chiếu mọi cách với oracle itertools.product trên chuỗi ngẫu nhiên (seed cố định)
rng = random.Random(20240784)
cases = 0
for _ in range(600):
    mode = rng.choice(["mixed", "letters", "digits_heavy"])
    n = rng.randint(1, 9)
    if mode == "mixed":
        pool = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
    elif mode == "letters":
        pool = "abcXYZ"
    else:
        pool = "0123456789a"
    s = "".join(rng.choice(pool) for _ in range(n))
    expected = oracle(s)
    for name, fn in methods.items():
        res = fn(s)
        assert sorted(res) == expected, (name, s)
    assert is_valid_answer(s, Solution().letterCasePermutation(s))
    cases += 1
print("đối chiếu", cases, "chuỗi ngẫu nhiên (n <= 9) với", len(methods), "cách: OK")

đối chiếu 600 chuỗi ngẫu nhiên (n <= 9) với 6 cách: OK


In [25]:
# 12.5 test lớn đúng ràng buộc: n = 12, toàn chữ cái -> 4096 kết quả
rng = random.Random(12)
big = {
    "12 chữ cái thường": "abcdefghijkl",
    "12 chữ cái lẫn hoa/thường": "aBcDeFgHiJkL",
    "11 chữ cái + 1 chữ số": "abcdefghijk7",
    "6 chữ cái + 6 chữ số": "a1b2c3d4e5f6",
    "toàn 12 chữ số": "123456789012",
    "ngẫu nhiên": "".join(rng.choice("abcXYZ0123") for _ in range(12)),
}
for title, s in big.items():
    k = sum(ch.isalpha() for ch in s)
    t0 = time.perf_counter()
    results = {name: sorted(fn(s)) for name, fn in methods.items() if name != "brute_force" or k >= 0}
    dt = time.perf_counter() - t0
    first = next(iter(results.values()))
    assert all(r == first for r in results.values()), title
    assert len(first) == 2 ** k and first == oracle(s)
    print(f"{title:26s} k = {k:2d}: {len(first):4d} kết quả, mọi cách khớp  ({dt * 1000:.0f} ms cho 6 cách)")
print("test lớn: OK")

12 chữ cái thường          k = 12: 4096 kết quả, mọi cách khớp  (17 ms cho 6 cách)
12 chữ cái lẫn hoa/thường  k = 12: 4096 kết quả, mọi cách khớp  (17 ms cho 6 cách)
11 chữ cái + 1 chữ số      k = 11: 2048 kết quả, mọi cách khớp  (12 ms cho 6 cách)
6 chữ cái + 6 chữ số       k =  6:   64 kết quả, mọi cách khớp  (7 ms cho 6 cách)
toàn 12 chữ số             k =  0:    1 kết quả, mọi cách khớp  (6 ms cho 6 cách)
ngẫu nhiên                 k =  6:   64 kết quả, mọi cách khớp  (5 ms cho 6 cách)
test lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Gán trực tiếp vào chuỗi (`s[i] = ...`)

Chuỗi trong Python **bất biến**; không sửa được từng ký tự.

In [26]:
s = "a1b2"
try:
    s[0] = "A"
except TypeError as e:
    print("SAI  : TypeError:", e)
t = list(s)
t[0] = "A"
print("ĐÚNG:", "".join(t))

SAI  : TypeError: 'str' object does not support item assignment
ĐÚNG: A1b2


**Vì sao sai:** kiểu `str` không hỗ trợ gán theo chỉ số. **Sửa:** đổi sang `list(s)`, sửa, rồi `"".join(...)`; hoặc dựng chuỗi mới bằng cắt lát `s[:i] + ch + s[i+1:]`.

### Lỗi 2: XOR `32` lên chữ số (không kiểm tra `isalpha`)

XOR `32` chỉ có nghĩa với chữ cái. Với chữ số cho ký tự lạ và tạo thêm nhánh vô nghĩa.

In [27]:
res = wrong_xor_digit("a1b")
print("SAI  : số chuỗi =", len(res), "(đúng 4); ví dụ có ký tự lạ:", [repr(x) for x in res][:4])
print("ĐÚNG:", sorted(Solution().letterCasePermutation("a1b")))
assert len(res) == 8

SAI  : số chuỗi = 8 (đúng 4); ví dụ có ký tự lạ: ["'a1b'", "'a1B'", "'a\\x11B'", "'a\\x11b'"]
ĐÚNG: ['A1B', 'A1b', 'a1B', 'a1b']


**Vì sao sai:** `chr(ord("1") ^ 32)` là ký tự điều khiển (mã 17), không phải chữ số; và ta rẽ hai nhánh ở cả chữ số nên có `2^n` chuỗi thay vì `2^k`. **Sửa:** `if t[i].isalpha():` trước khi đổi và rẽ nhánh thứ hai.

### Lỗi 3: Không rẽ hai nhánh (chỉ đổi một chiều)

Chỉ chuyển thành chữ hoa (hoặc chỉ chữ thường) cho ra **một** chuỗi.

In [28]:
print("SAI  :", wrong_no_branch("a1b"))
print("ĐÚNG:", sorted(Solution().letterCasePermutation("a1b")))

SAI  : ['A1B']
ĐÚNG: ['A1B', 'A1b', 'a1B', 'a1b']


**Vì sao sai:** thiếu nhánh "giữ nguyên" nên không có tổ hợp lẫn hoa/thường. **Sửa:** với mỗi chữ cái, đi cả hai nhánh.

### Lỗi 4: Thêm chính `list` đang sửa vào kết quả

Nếu `ans.append(t)` mà không `join`/sao chép, mọi phần tử của `ans` cùng trỏ **một** list `t` đang bị thay đổi.

In [29]:
def alias_demo(s):
    t = list(s)
    ans = []
    def dfs(i):
        if i == len(t):
            ans.append(t)               # SAI: cùng một list
            return
        dfs(i + 1)
        if t[i].isalpha():
            t[i] = chr(ord(t[i]) ^ 32)
            dfs(i + 1)
    dfs(0)
    return ans

res = alias_demo("ab")
print("SAI  :", ["".join(x) for x in res], "(4 phần tử giống hệt nhau)")
print("ĐÚNG:", sorted(Solution().letterCasePermutation("ab")))
assert len({"".join(x) for x in res}) == 1

SAI  : ['Ab', 'Ab', 'Ab', 'Ab'] (4 phần tử giống hệt nhau)
ĐÚNG: ['AB', 'Ab', 'aB', 'ab']


**Vì sao sai:** `ans[0] is ans[1] is ...`, và cuối cùng `t` mang trạng thái của lần ghi cuối. **Sửa:** `ans.append("".join(t))` (tạo chuỗi mới) hoặc `t[:]` (sao chép).

### Lỗi 5: Thử mặt nạ cho cả chữ số mà quên khử trùng

Lật chữ số không đổi gì, nên có nhiều mặt nạ cho cùng một chuỗi ⇒ kết quả **trùng lặp**.

In [30]:
r = wrong_dup("a1b")
print("SAI  :", len(r), "chuỗi, trong đó khác nhau:", len(set(r)))
print("ĐÚNG:", len(Solution().letterCasePermutation("a1b")), "chuỗi")
assert len(r) == 8 and len(set(r)) == 4

SAI  : 8 chuỗi, trong đó khác nhau: 4
ĐÚNG: 4 chuỗi


**Vì sao sai:** `a1b` có `2^3 = 8` mặt nạ nhưng chỉ `2^2 = 4` chuỗi khác nhau. **Sửa:** chỉ đặt công tắc cho chữ cái (Cách 4) hoặc dùng `set` (Cách 1).

### Lỗi 6: Giả định input toàn chữ thường

Nếu input chứa chữ hoa, chỉ "đổi thành hoa" sẽ không bao giờ sinh dạng thường. Hãy dùng **cả** `lower` và `upper`, hoặc **lật** (`swapcase`/XOR `32`) từ trạng thái hiện tại.

In [31]:
print("SAI  ('A1B'):", sorted(set(wrong_only_from_lower("A1B"))))
print("ĐÚNG ('A1B'):", sorted(Solution().letterCasePermutation("A1B")))

SAI  ('A1B'): ['A1B']
ĐÚNG ('A1B'): ['A1B', 'A1b', 'a1B', 'a1b']


**Vì sao sai:** mọi chữ đã là hoa, "đổi thành hoa" không thay đổi gì. **Sửa:** dùng `(lower, upper)` cho mỗi chữ cái hoặc lật.

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra:**

- "Liệt kê **mọi** biến thể" khi `n` rất nhỏ (`<= 12`): **quay lui / bitmask**.
- Mỗi phần tử có **một số lựa chọn độc lập** (`2` cho chữ cái, `1` cho chữ số): **tích Descartes** (`itertools.product`), số kết quả là tích số lựa chọn.
- Đề cho phép **thứ tự bất kỳ** → test bằng `sorted` hoặc bằng hàm kiểm tra tính chất.

**Mẹo:**

1. Dùng `swapcase`/XOR `32` để **lật** thay vì tính đường đi thủ công, nhưng chỉ với `isalpha()`.
2. Chuỗi bất biến: làm việc trên `list`, đến cuối mới `join`.
3. Chỉ phân nhánh khi **có lựa chọn thật**; chữ số đi thẳng ⇒ tránh trùng.
4. Đếm trước số kết quả `2^k` để kiểm tra nhanh độ dài đáp án.
5. Khi chỉ **duyệt** (không lưu): mã Gray cho phép cập nhật `O(1)` mỗi bước.
6. `list comprehension` `[p + c for p in res for c in options]` là cách ngắn gọn để "nhân" tập kết quả (BFS).

## 15. Biến thể

### 15.1. `itertools.product` (cách ngắn gọn nhất)

Mỗi ký tự cho tuple lựa chọn; `product(*choices)` sinh mọi tổ hợp:

In [32]:
def with_product(s):
    return ["".join(p) for p in product(*[(c.lower(), c.upper()) if c.isalpha() else (c,) for c in s])]

for s, expected in examples:
    assert sorted(with_product(s)) == sorted(expected)
print(with_product("a1b2"))

['a1b2', 'a1B2', 'A1b2', 'A1B2']


### 15.2. Bộ sinh `yield` (sinh từng kết quả, tiết kiệm bộ nhớ)

Nếu chỉ cần duyệt kết quả một lần, dùng **generator**: không phải giữ toàn bộ danh sách `2^k` chuỗi trong bộ nhớ.

In [33]:
def gen_case_permutations(s):
    t = list(s)
    def dfs(i):
        if i == len(t):
            yield "".join(t)
            return
        yield from dfs(i + 1)
        if t[i].isalpha():
            t[i] = chr(ord(t[i]) ^ 32)
            yield from dfs(i + 1)
    yield from dfs(0)

g = gen_case_permutations("a1b2")
print(next(g), next(g))                     # lấy từng kết quả theo yêu cầu
assert sorted(gen_case_permutations("a1b2")) == sorted(examples[0][1])

a1b2 a1B2


### 15.3. Truy cập trực tiếp kết quả thứ `idx` (theo thứ tự sắp xếp)

Trong thứ tự ASCII, chữ số < chữ hoa < chữ thường. Các kết quả được sắp xếp tương ứng với mặt nạ nhị phân `idx` đọc từ **chữ cái đầu (bit cao nhất) đến chữ cái cuối (bit thấp nhất)**, với bit `0` = chữ hoa, bit `1` = chữ thường. Do đó có thể lấy kết quả thứ `idx` (đếm từ `0`) **mà không sinh hết**. Đây cũng là bài tập 5.

In [34]:
def kth_case(s, idx):
    pos = [i for i, ch in enumerate(s) if ch.isalpha()]
    k = len(pos)
    t = list(s)
    for j, p in enumerate(pos):
        bit = idx >> (k - 1 - j) & 1          # chữ cái đầu là bit cao nhất
        t[p] = s[p].lower() if bit else s[p].upper()
    return "".join(t)

s = "a1b2"
print([kth_case(s, i) for i in range(4)])
assert [kth_case(s, i) for i in range(4)] == sorted(Solution().letterCasePermutation(s))
rng = random.Random(153)
for _ in range(200):
    s = "".join(rng.choice("abXY019") for _ in range(rng.randint(1, 9)))
    k = sum(ch.isalpha() for ch in s)
    ordered = oracle(s)
    for idx in range(2 ** k):
        assert kth_case(s, idx) == ordered[idx]
print("kth_case khớp danh sách đã sắp xếp trên 200 chuỗi ngẫu nhiên")

['A1B2', 'A1b2', 'a1B2', 'a1b2']
kth_case khớp danh sách đã sắp xếp trên 200 chuỗi ngẫu nhiên


### 15.4. Các bài cùng họ

- **LeetCode 78 — Subsets:** mỗi phần tử có 2 lựa chọn chọn/không chọn (cũng `2^n`).
- **LeetCode 17 — Letter Combinations of a Phone Number:** mỗi chữ số có 3 hoặc 4 lựa chọn: tích Descartes.
- **LeetCode 22 — Generate Parentheses**, **46 — Permutations:** liệt kê bằng quay lui.
- **LeetCode 89 — Gray Code:** sinh mã Gray (Cách 5).
- **LeetCode 320 — Generalized Abbreviation:** mỗi ký tự giữ hoặc rút gọn.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mặt nạ mọi ký tự + `set` | thử `2^n` mặt nạ, khử trùng | `O(n * 2^n)` | `O(n * 2^k)` | baseline; chậm khi nhiều chữ số |
| 2. DFS quay lui | rẽ hai nhánh ở chữ cái | `O(n * 2^k)` | `O(n)` | **cách nộp chuẩn** |
| 3. Mở rộng dần (BFS) | nhân danh sách với tập lựa chọn từng ký tự | `O(n * 2^k)` | `O(n * 2^k)` | code rất ngắn, dễ viết |
| 4. Mặt nạ bit trên chữ cái | `2^k` mặt nạ, bit = hoa/thường | `O(n * 2^k)` | `O(n)` | không đệ quy, cần truy cập theo chỉ số |
| 5. Mã Gray | mỗi bước lật đúng một chữ cái | `O(n * 2^k)`, bước tiếp `O(1)` | `O(n)` | khi chỉ duyệt, không cần lưu |

Kết luận: dùng Cách 2 (hoặc Cách 3 cho ngắn); nhớ rằng độ phức tạp thấp nhất có thể là `Ω(n * 2^k)` vì kích thước kết quả.

## 17. Bài tập vận dụng

Làm từ dễ đến khó. Đáp án ở **mục 19**. Các cell TODO **chưa được gọi**; bạn tự điền rồi tự chạy.

### Bài 1 (đoán output)

Không chạy code, đoán kết quả:

```python
res = Solution().letterCasePermutation("a1B")
print(len(res))
print(sorted(res))
```

### Bài 2 (tính tay)

Với `s = "ab"`: liệt kê tất cả chuỗi kết quả. Rồi với `s = "9x9y9"`: có bao nhiêu kết quả và chúng là gì?

### Bài 3 (điền TODO)

Điền chỗ trống của Cách 3 (mở rộng dần):

In [35]:
def ex3_bfs(s):
    res = [""]
    for ch in s:
        if ch.isalpha():
            options = (..., ...)                 # TODO 1: hai lựa chọn cho chữ cái
        else:
            options = (...,)                     # TODO 2: một lựa chọn cho chữ số
        res = [... for p in res for c in options]   # TODO 3: nối tiền tố p với lựa chọn c
    return res

# Khi đã điền xong, bỏ dấu # ở dòng sau để thử:
# assert sorted(ex3_bfs("a1b2")) == sorted(["a1b2", "a1B2", "A1b2", "A1B2"])

### Bài 4 (tự viết lại)

Viết lại **Cách 4 (mặt nạ bit trên các chữ cái)** từ đầu, không nhìn bài giảng. Gợi ý: lấy danh sách vị trí chữ cái; duyệt `mask` từ `0` đến `2^k - 1`; bit `j` quyết định chữ cái thứ `j` là hoa hay thường.

In [36]:
def ex4_bitmask(s):
    # TODO: tự viết lời giải mặt nạ bit trên các chữ cái, trả về list chuỗi
    ...

# Khi đã viết xong, bỏ dấu # để thử:
# assert sorted(ex4_bitmask("3z4")) == ["3Z4", "3z4"]

### Bài 5 (mini challenge, biến thể)

Viết `kth_case(s, idx)` trả về chuỗi kết quả đứng thứ `idx` (đếm từ `0`) **khi các kết quả được sắp xếp tăng dần** (thứ tự ASCII: chữ số < chữ hoa < chữ thường), **không sinh hết** danh sách. Gợi ý: chữ cái đầu tiên là bit cao nhất của `idx`; bit `0` là chữ hoa, bit `1` là chữ thường.

In [37]:
def ex5_kth_case(s, idx):
    # TODO: trả về chuỗi kết quả thứ idx trong danh sách đã sắp xếp (không sinh hết)
    ...

# Khi đã viết xong, thử bằng test ở mục 19

## 18. Gợi ý 3 mức

**Bài 3**

- Mức 1 (nhẹ): chữ cái có hai dạng; chữ số chỉ một dạng.
- Mức 2 (vừa): `ch.lower()` và `ch.upper()` cho chữ cái; chữ số thì `ch`.
- Mức 3 (gần đáp án): `options = (ch.lower(), ch.upper())`; `options = (ch,)`; `res = [p + c for p in res for c in options]`.

**Bài 4**

- Mức 1: tìm vị trí các chữ cái: `pos = [i for i, ch in enumerate(s) if ch.isalpha()]`.
- Mức 2: với mỗi `mask in range(1 << len(pos))`, sao chép `t = list(s)` rồi gán lại từng chữ cái theo bit.
- Mức 3: `t[p] = s[p].upper() if mask >> j & 1 else s[p].lower()`, cuối cùng `"".join(t)`.

**Bài 5**

- Mức 1: chuỗi nhỏ nhất theo ASCII có mọi chữ cái là **hoa** (vì `'A' < 'a'`).
- Mức 2: kết quả thứ `idx` ứng với mặt nạ `idx` viết nhị phân `k` bit; bit cao nhất ứng với chữ cái đầu tiên.
- Mức 3: với `j`-th chữ cái (từ `0`): `bit = idx >> (k - 1 - j) & 1`; bit `1` → thường, bit `0` → hoa.

## 19. Đáp án tham khảo

**Bài 1:** `"a1B"` có `2` chữ cái nên `4` kết quả. Sắp xếp (ASCII: `'A' < 'B' < 'a' < 'b'`): `['A1B', 'A1b', 'a1B', 'a1b']`. In ra `4` rồi danh sách trên.

**Bài 2:** `"ab"` → `['AB', 'Ab', 'aB', 'ab']`. `"9x9y9"` có `2` chữ cái nên `4` kết quả: `['9X9Y9', '9X9y9', '9x9Y9', '9x9y9']` (chữ số giữ nguyên).

In [38]:
# Bài 1
res = Solution().letterCasePermutation("a1B")
assert len(res) == 4 and sorted(res) == ["A1B", "A1b", "a1B", "a1b"]
print("Bài 1:", len(res), sorted(res))

# Bài 2
assert sorted(Solution().letterCasePermutation("ab")) == ["AB", "Ab", "aB", "ab"]
assert sorted(Solution().letterCasePermutation("9x9y9")) == ["9X9Y9", "9X9y9", "9x9Y9", "9x9y9"]
print("Bài 2: OK")

# Bài 3
def ex3_answer(s):
    res = [""]
    for ch in s:
        options = (ch.lower(), ch.upper()) if ch.isalpha() else (ch,)
        res = [p + c for p in res for c in options]
    return res

# Bài 4
def ex4_answer(s):
    pos = [i for i, ch in enumerate(s) if ch.isalpha()]
    out = []
    for mask in range(1 << len(pos)):
        t = list(s)
        for j, p in enumerate(pos):
            t[p] = s[p].upper() if mask >> j & 1 else s[p].lower()
        out.append("".join(t))
    return out

# Bài 5
def ex5_answer(s, idx):
    pos = [i for i, ch in enumerate(s) if ch.isalpha()]
    k = len(pos)
    t = list(s)
    for j, p in enumerate(pos):
        t[p] = s[p].lower() if idx >> (k - 1 - j) & 1 else s[p].upper()
    return "".join(t)

assert sorted(ex3_answer("a1b2")) == sorted(examples[0][1])
assert sorted(ex4_answer("3z4")) == ["3Z4", "3z4"]
rng = random.Random(19)
for _ in range(300):
    s = "".join(rng.choice("abcXYZ0123") for _ in range(rng.randint(1, 9)))
    exp = oracle(s)
    assert sorted(ex3_answer(s)) == sorted(ex4_answer(s)) == exp
    k = sum(ch.isalpha() for ch in s)
    for idx in range(2 ** k):
        assert ex5_answer(s, idx) == exp[idx]
print("Bài 3, 4, 5: khớp oracle trên 300 chuỗi ngẫu nhiên")

Bài 1: 4 ['A1B', 'A1b', 'a1B', 'a1b']
Bài 2: OK


Bài 3, 4, 5: khớp oracle trên 300 chuỗi ngẫu nhiên


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "liệt kê mọi biến thể chữ hoa/thường" (DFS):**

```python
class Solution:
    def letterCasePermutation(self, s: str) -> List[str]:
        def dfs(i: int) -> None:
            if i >= len(t):
                ans.append("".join(t))
                return
            dfs(i + 1)                         # giữ nguyên
            if t[i].isalpha():
                t[i] = chr(ord(t[i]) ^ 32)     # lật hoa/thường
                dfs(i + 1)

        t = list(s)
        ans = []
        dfs(0)
        return ans
```

**Template ngắn (BFS):**

```python
res = [""]
for ch in s:
    options = (ch.lower(), ch.upper()) if ch.isalpha() else (ch,)
    res = [p + c for p in res for c in options]
return res
```

**Checklist trước khi submit:**

- [ ] Chuỗi bất biến: đã dùng `list(s)` và `"".join(t)` khi ghi kết quả?
- [ ] Chỉ rẽ **hai nhánh** ở **chữ cái** (`isalpha`), chữ số đi thẳng?
- [ ] Chỉ XOR `32` (hoặc `swapcase`) với chữ cái?
- [ ] Kết quả có đúng `2^k` chuỗi, không trùng?
- [ ] Input có thể chứa chữ hoa; đã xử lý cả hai chiều?
- [ ] Đã thử `"7"`, `"a"`, `"Z"`, toàn chữ số, `n = 12` toàn chữ cái?
- [ ] Test so sánh `sorted(...)` vì thứ tự bất kỳ?
- [ ] Time `O(n * 2^k)`, Space `O(n)` (không tính kết quả)?

**Ghi nhớ một câu:** *"Mỗi chữ cái là một công tắc hoa/thường, chữ số đứng yên: có 2^k kết quả; liệt kê bằng quay lui, mặt nạ bit hoặc nhân danh sách."*